# PatchCore Quickstart — MVTec AD
Train-free defect detection & localization in a few minutes on a free Colab **T4 GPU**.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone the repo & install

In [ ]:
# If you pushed this to GitHub, replace the URL below.
# On Colab you can also just upload the repo folder.
!git clone https://github.com/mtaha-ai/industrial-anomaly-detection.git
%cd industrial-anomaly-detection
!pip install -q -e .

## 2. Download one category
The full dataset is ~4.9 GB. For a quickstart we only need `bottle`.
On **Kaggle** you can skip this and point `--root` at `/kaggle/input/mvtec-ad`.

In [ ]:
# Full download (all categories). Comment out if you already have the data.
!python scripts/download_data.py --root ./data/mvtec_ad

## 3. Fit PatchCore on the good images and evaluate

In [ ]:
import torch
from torch.utils.data import DataLoader
from iad import PatchCore, MVTecDataset, evaluate
from iad.utils.seed import set_seed

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
root = './data/mvtec_ad'
category = 'bottle'

train_ds = MVTecDataset(root, category, 'train')
test_ds  = MVTecDataset(root, category, 'test')
train_loader = DataLoader(train_ds, batch_size=8)
test_loader  = DataLoader(test_ds, batch_size=8)

model = PatchCore(backbone='wide_resnet50_2', coreset_ratio=0.1, device=device)
model.fit(train_loader)
metrics = evaluate(model, test_loader)
print(metrics)

## 4. Visualize anomaly heatmaps
Input | Ground truth | Heatmap | Overlay.

In [ ]:
import matplotlib.pyplot as plt
from iad.visualize import save_comparison_grid

defect_idx = [i for i,l in enumerate(test_ds.labels) if l==1][:4]
images, maps, masks, titles = [], [], [], []
for i in defect_idx:
    s = test_ds[i]
    p = model.predict(s['image'].unsqueeze(0))
    images.append(s['image']); maps.append(p.anomaly_maps[0].numpy())
    masks.append(s['mask']);   titles.append(s['defect_type'])

path = save_comparison_grid(images, maps, masks, 'results/figures/bottle_demo.png', titles=titles)
from IPython.display import Image as IPImage
IPImage(filename=str(path))

That's it — a fully training-free detector with pixel-level localization.
See `02_run_benchmark.ipynb` to reproduce the full 15-category table.